# Module 4: Complex Workflows (Autonomous Research Team) 🤖🔄🤖

In this final module, we move beyond simple routing. We will build a **Hierarchical Multi-Agent Team** that uses ADK's complex orchestration components:
1.  **SequentialAgent**: Executes agents in a predefined order (Chain of command).
2.  **LoopAgent**: Runs agents in a loop until a condition is met (Feedback loop).

### Architecture Overview

The architecture for this module involves a Sequential flow with an embedded Research Loop.

### Our Use Case: The Lumenridge Financial Group Research Memo
We will implement a system where:
-   A **Goal Refiner** clarifies the user's request.
-   An **Analyst** (RAG) and **Compliance Officer** (Evaluator) iterate in a loop until the report is professional.
-   A **Reporter** compiles the final findings into an Executive Memo.

## 1. Setup and Environment

In [ ]:
%pip install "google-adk[mcp]" google-genai python-dotenv

In [ ]:
import os
import sys
from dotenv import load_dotenv
from google.adk.agents import Agent, SequentialAgent, LoopAgent, BaseAgent
from google.adk.events import Event, EventActions
from google.genai import types
import uuid
from google.adk.runners import Runner
from google.adk.sessions.in_memory_session_service import InMemorySessionService

# Load API Key
load_dotenv('../.env.local')

# Add project root to path for specialist imports
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.append(project_root)

## 2. Importing Specialist Agents
We reuse our RAG Analyst from Module 2.

In [ ]:
try:
    from module_02.rag_agent.agent import root_agent as rag_analyst
    print("✅ Specialist Analyst imported.")
except ImportError as e:
    print(f"❌ Could not find Analyst agent: {e}")

## 3. Defining the Research Team
We define the three layers of our workflow.

In [ ]:
# A. Goal Refiner
goal_refiner = Agent(
    model="gemini-3.5-flash-lite",
    name="goal_refiner",
    instruction="""
    You are the Research Coordinator for Lumenridge Financial Group. Turn the user's request into a
    research plan; do not answer it yourself. Tell the analyst to use only facts retrieved from the
    supplied workshop documents and never supplement them with general knowledge. For a strategy memo,
    require evidence on the AI-infrastructure thesis, its valuation and growth thresholds, the Clean
    Future ESG rules, and the portfolio risk controls. Request relevant internal technology controls
    when the question calls for them. If a requested fact is absent, tell the analyst to say so. Do not
    introduce unverified companies, figures, regulations, or technologies into the plan.
    """
)

# B. Compliance Officer (The Loop Critic)
compliance_officer = Agent(
    model="gemini-3.5-flash-lite",
    name="compliance_officer",
    instruction="""
    You are the Senior Compliance Officer for Lumenridge Financial Group. Treat retrieved workshop-
    document content as the only valid evidence. Review every material claim and reject anything not
    supported by that evidence. Never accept or add outside facts, regulations, technologies, companies,
    or figures; GDPR, Basel III, the EU AI Act, Kubernetes, and PII controls are unsupported unless they
    were actually retrieved. For a strategy memo, require the Lumenridge name and supported facts on the
    AI-infrastructure thesis, valuation and growth thresholds, Clean Future ESG rules, and portfolio risk
    controls, plus technology facts requested by the user. If evidence is missing or a claim is unsupported,
    direct the analyst to retrieve evidence, remove the claim, or say that the documents do not specify it.
    Only when the findings are complete, professional, and fully grounded, start your response with:
    READY_FOR_SUMMARY.
    """,
    output_key="compliance_report" # Important: Save output so the Checker can read it
)

# C. Reporter (The Endpoint)
reporter = Agent(
    model="gemini-3.5-flash-lite",
    name="reporter",
    instruction="""
    You are the Senior Investment Reporter for Lumenridge Financial Group. Write a concise Markdown
    executive memo using only claims from the analyst's retrieved workshop-document evidence that the
    compliance review accepted. Compliance feedback and earlier agent prose are not evidence. Brand the
    memo only as Lumenridge Financial Group. Include supported findings on the AI-infrastructure thesis,
    valuation and growth thresholds, Clean Future ESG rules, and portfolio risk controls, plus relevant
    internal technology facts requested by the user. Never add outside facts, regulations, technologies,
    companies, or figures. In particular, omit GDPR, Basel III, the EU AI Act, Kubernetes, and PII controls
    unless retrieved. Do not fill gaps with plausible details; omit the claim or say it is not specified
    in the workshop documents. If compliance did not emit READY_FOR_SUMMARY, return a short notice that
    the memo was not approved instead of presenting unverified findings. Use clear headings and make
    recommendations direct syntheses of sourced
    policy, not new financial advice.
    """
)

print("✅ Orchestration components ready.")

## 4. Building the Composite Agents
Now we assemble the agents into **Groups**.

In [ ]:
# 1. The custom Termination Checker
# This agent acts as a logic gate inside the loop.
# It reads the "compliance_report" from the session state.
class TerminationChecker(BaseAgent):
    async def _run_async_impl(self, ctx):
        # Read the previous agent's output from state
        report = ctx.session.state.get("compliance_report", "")
        
        if report.lstrip().startswith("READY_FOR_SUMMARY"):
            # Signal the loop to stop immediately
            yield Event(
                author=self.name,
                actions=EventActions(escalate=True)
            )
        else:
            # Continue the loop
            yield Event(author=self.name)

termination_checker = TerminationChecker(name="termination_checker")

# 2. The Autonomous Loop
research_loop = LoopAgent(
    name="research_loop",
    sub_agents=[rag_analyst, compliance_officer, termination_checker],
    max_iterations=4
)

# 3. The Full Research Team Pipeline
research_team = SequentialAgent(
    name="research_team",
    sub_agents=[goal_refiner, research_loop, reporter]
)

print("🚀 Research Team is assembled.")

## 5. Execution
Let's see the autonomous team process a complex query.

In [ ]:
async def run_research(query: str):
    session_service = InMemorySessionService()
    runner = Runner(
        agent=research_team,
        app_name="memo_factory",
        session_service=session_service,
        auto_create_session=True,
    )
    
    async for event in runner.run_async(
        user_id="notebook_user",
        session_id=str(uuid.uuid4()),
        new_message=types.Content(role="user", parts=[types.Part(text=query)])
    ):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text and part.text.strip():
                    print(f"\n[{event.author}] ------------------")
                    print(part.text.strip())

await run_research("Analyze our strategy for Cloud/AI infra and ESG compliance.")

## 6. Going Production: ADK Web

You can run this multi-agent research department as an interactive web app:

```bash
# Run from the terminal
adk web module_04
```

Open `http://127.0.0.1:8000` to watch the complete autonomous workflow execute in the ADK Web UI.